# STT – Version 2 : semi temps réel + arrêt vocal

Cette version améliore la V1 en ajoutant :

- une boucle semi temps réel par segments de 2 secondes,
- un filtrage des segments silencieux,
- un arrêt vocal via le mot-clé **"stop"** dans la transcription.

La V1 ne gérait qu'un enregistrement unique et une transcription simple.


In [2]:
import os
import shutil

# On ajoute le dossier ffmpeg au PATH du processus Python
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"

print("ffmpeg trouvé par Python ? ->", shutil.which("ffmpeg"))


ffmpeg trouvé par Python ? -> C:\ffmpeg\ffmpeg-8.0-essentials_build\bin\ffmpeg.EXE


##  Installation des dépendances Python

Les principales bibliothèques utilisées sont :
- `whisper` : modèle de reconnaissance vocale pré-entraîné.
- `sounddevice` : enregistrement audio depuis le micro.
- `scipy` + `numpy` : manipulation et sauvegarde des signaux audio.

Sur cette machine, ces paquets ont été installés via `pip` :

```bash
pip install openai-whisper sounddevice scipy


In [ ]:
# À exécuter seulement si les libs ne sont pas encore installées sur la machine
# !pip install openai-whisper sounddevice scipy --quiet

##  Imports et configuration de base

Dans cette cellule, nous :
- importons les modules nécessaires (`whisper`, `sounddevice`, `numpy`, etc.),
- affichons le dossier de travail courant,
- listons les fichiers présents (utile pour vérifier la présence de `mon_audio.wav` après enregistrement).


In [3]:
import whisper
import sounddevice as sd
from scipy.io.wavfile import write
import numpy as np

import os

print("Dossier courant :", os.getcwd())
print("Fichiers présents :", os.listdir())


Dossier courant : d:\Desktop\voix-langue-des-signes\stt
Fichiers présents : ['mon_audio.wav', 'stt_speech_to_text.ipynb']


##  Chargement du modèle Whisper

Nous utilisons le modèle **`small`** de Whisper, qui offre un bon compromis entre :
- temps de calcul (raisonnable sur CPU),
- qualité de la transcription.

Remarque :
- Le warning `FP16 is not supported on CPU; using FP32 instead` est **normal** sur CPU :
  cela signifie simplement que le modèle utilisera des flottants 32 bits au lieu de 16 bits.


In [4]:
model = whisper.load_model("small")
print("Modèle Whisper chargé.")


Modèle Whisper chargé.


## Version "semi temps réel" avec segments de 2 secondes

Objectif :
- Enregistrer la voix en continu, par segments de 2 secondes.
- Transcrire chaque segment dès qu'il est capturé.
- Afficher progressivement la transcription (segment par segment).
- Préparer un point d'accroche pour envoyer le texte vers le module "Texte → Signes".

Remarque :
- On n'enregistre plus dans un fichier `.wav` à chaque fois : on passe directement 
  le tableau NumPy au modèle Whisper, ce qui réduit les dépendances à `ffmpeg`.
- Cette approche est "semi temps réel" : latence ≈ durée segment (2 s) + temps de calcul.


In [7]:
import os
import shutil
import numpy as np
import sounddevice as sd
import whisper

# (Optionnel) S'assurer que ffmpeg est dans le PATH, si un jour tu veux aussi transcrire des fichiers
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"
print("ffmpeg trouvé par Python ? ->", shutil.which("ffmpeg"))

# Paramètres audio
FS = 16000           # fréquence d'échantillonnage
SEGMENT_DURATION = 2 # durée d'un segment en secondes
N_SAMPLES = FS * SEGMENT_DURATION

# Chargement du modèle Whisper
# Tu peux tester "base" (plus rapide) ou rester sur "small"
model = whisper.load_model("small")
print("Modèle Whisper chargé.")


ffmpeg trouvé par Python ? -> C:\ffmpeg\ffmpeg-8.0-essentials_build\bin\ffmpeg.EXE
Modèle Whisper chargé.


## Boucle de capture et transcription par segments de 2 s

Stratégie :
- On utilise `sounddevice.rec` pour enregistrer un bloc de `SEGMENT_DURATION` secondes.
- On obtient un tableau NumPy `audio` de taille `(N_SAMPLES, 1)`, en float32, à 16 kHz.
- On passe ce tableau directement à `model.transcribe`, sans sauvegarde disque.
- On applique un seuil d'énergie simple pour ignorer (optionnellement) les segments silencieux.

Arrêt :
- La boucle tourne jusqu'à une interruption clavier (`Ctrl+C`).


In [8]:
def capture_and_transcribe_loop(language="fr", energy_threshold=0.01):
    """
    Enregistre et transcrit en continu, par segments de 2 secondes.
    
    - language : langue pour Whisper (ex: "fr")
    - energy_threshold : seuil moyen absolu sous lequel on considère le segment comme "silencieux"
    """
    print("Démarrage de la boucle semi temps réel.")
    print("Parle par petites phrases. Ctrl+C pour arrêter.\n")
    
    segment_idx = 0
    try:
        while True:
            print(f"\n[Segment {segment_idx}] Enregistrement…")
            audio = sd.rec(int(N_SAMPLES), samplerate=FS, channels=1, dtype="float32")
            sd.wait()  # on attend la fin du segment

            # On écrase en 1D (mono)
            audio_mono = audio[:, 0]

            # Mesure simple de l'énergie moyenne pour détecter le silence
            energy = float(np.mean(np.abs(audio_mono)))
            print(f"Énergie moyenne du segment : {energy:.5f}")

            if energy < energy_threshold:
                print("[Segment ignoré : trop silencieux]")
                segment_idx += 1
                continue

            # Transcription avec Whisper (sans fp16 sur CPU)
            print("[Transcription en cours…]")
            result = model.transcribe(audio_mono, language=language, fp16=False)
            text = result["text"].strip()

            if text:
                print(f"[Segment {segment_idx}] Texte reconnu : {text}")
                # 👉 ICI tu pourras appeler le module "texte -> gloss -> signes"
                # par ex: send_to_sign_module(text)
            else:
                print(f"[Segment {segment_idx}] Aucun texte reconnu.")

            segment_idx += 1

    except KeyboardInterrupt:
        print("\nArrêt de la boucle par l'utilisateur (Ctrl+C).")


## Lancement de la démo semi temps réel

Exécution :
- Lancer la cellule suivante.
- Parler par petites phrases (2–3 secondes max).
- Attendre que le texte apparaisse segment par segment.
- Pour arrêter : utiliser `Ctrl + C` dans le terminal / kernel.

Note :
- Plus le modèle est gros, plus la latence de transcription sera longue.
- Si c'est trop lent, tu peux tester `whisper.load_model("base")` ou `whisper.load_model("tiny")`.


In [11]:
capture_and_transcribe_loop(language="fr", energy_threshold=0.01)


Démarrage de la boucle semi temps réel.
Parle par petites phrases. Ctrl+C pour arrêter.


[Segment 0] Enregistrement…
Énergie moyenne du segment : 0.01627
[Transcription en cours…]
[Segment 0] Texte reconnu : et le bonjour

[Segment 1] Enregistrement…
Énergie moyenne du segment : 0.01706
[Transcription en cours…]
[Segment 1] Texte reconnu : J' bisher use dialecticale

[Segment 2] Enregistrement…
Énergie moyenne du segment : 0.00081
[Segment ignoré : trop silencieux]

[Segment 3] Enregistrement…
Énergie moyenne du segment : 0.01629
[Transcription en cours…]
[Segment 3] Texte reconnu : Je suis...

[Segment 4] Enregistrement…
Énergie moyenne du segment : 0.00266
[Segment ignoré : trop silencieux]

[Segment 5] Enregistrement…
Énergie moyenne du segment : 0.02440
[Transcription en cours…]
[Segment 5] Texte reconnu : Je suis... Je suis...

[Segment 6] Enregistrement…
Énergie moyenne du segment : 0.00187
[Segment ignoré : trop silencieux]

[Segment 7] Enregistrement…
Énergie moyenne du segmen

## Améliorations possibles pour une V3

- Optimiser les performances (modèle `base` ou `tiny` pour réduire la latence),
- Intégrer directement la sortie au pipeline "texte → gloss → signes",
- Gérer plusieurs langues (français + anglais).
